# Genome Garden in Colab

Watch evolution happen in your browser, with nothing to install. Python runs on Google's computer, and the garden opens in a new tab.

Run the cells below in order: click a cell, then press the ▶ button on its left (or Shift+Enter). You need to be signed in to a Google account.

## 1. Download Genome Garden
This takes a few seconds.

In [ ]:
import os, subprocess, sys

BRANCH = "feat/colab"    # back to main before merging
HOME = "/content/genome-garden"
if not os.path.exists(HOME):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, "https://github.com/sbbis25/genome-garden", HOME], check=True)
else:    # already downloaded in this session: fetch the latest of BRANCH, keeping your workshop.py edits
    git = lambda *a: subprocess.run(["git", "-C", HOME] + list(a)).returncode
    if git("fetch", "-q", "--depth", "1", "origin", BRANCH) or git("checkout", "-q", "-B", BRANCH, "FETCH_HEAD"):
        print("Could not download updates, so the copy already here is used.")
os.chdir(HOME)
sys.path.insert(0, HOME)

# python keeps the first garden it loaded, so an update needs a fresh session to take effect
commit = subprocess.run(["git", "-C", HOME, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if "garden" not in sys.modules:
    os.environ["GG_LOADED"] = commit
elif os.environ.get("GG_LOADED") != commit:
    raise SystemExit("Genome Garden was updated. Choose Runtime → Restart session, then Runtime → Run all.")
print("Ready.")

## 2. Start the world
The garden appears below this cell and keeps running while this notebook is open. For more room, open the link under it in its own tab.

In [ ]:
from garden.server import start
from google.colab import output

# 0.0.0.0 lets colab's proxy reach the garden; this machine belongs to google, not you
port = start(HOME + "/workshop.py", host="0.0.0.0")
output.serve_kernel_port_as_iframe(port, height=820)
print("Want it bigger? Open it in its own tab:", output.eval_js("google.colab.kernel.proxyPort(%d)" % port))

## 3. Write your own rules
Run this cell to open `workshop.py`, the one file you edit, in an editor on the right. Uncomment an example (delete the `# ` at the start of each line), then save with **Ctrl+S** (**Cmd+S** on a Mac). The running world picks up your change within a second. Mistakes show up in the garden's **Code** tab with the line number, and the world keeps running.

You do not need to run cell 2 again after you edit.

In [ ]:
from google.colab import files

files.view(HOME + "/workshop.py")

## 4. Keep your code
Your `workshop.py` lives on Google's computer and is deleted when the session ends. Run this cell to download a copy.

In [ ]:
from google.colab import files

files.download(HOME + "/workshop.py")

## If something goes wrong
- **The garden tab stopped updating:** Colab closes sessions that sit idle for a while, or when you close the notebook. Choose **Runtime → Run all** and click the new link.
- **A blank box or "404 Not Found":** your browser may be blocking Colab's cookies. Use a normal (not private) Chrome window. If it still fails, choose **Runtime → Disconnect and delete runtime**, then **Runtime → Run all**.
- **It feels choppy:** every frame travels through Google's servers, so it is smoother when run on your own laptop with `python run.py`.

## Connection test
Only if the garden will not show up. Run step 1 first, then this cell. It starts three small test servers and shows each one below, with a link to open it in its own tab. Wait a few seconds, try the links, then **run this cell again** to list what each server received. Send the whole output to an organizer.

In [ ]:
import subprocess, sys, threading, time, urllib.request
from http.server import ThreadingHTTPServer
from google.colab import output
from garden.server import Sim, make_handler
from garden.world import World

if "diag" not in globals():
    diag = {"B": [], "C": []}

    def logging_garden(host, port, seen):
        sim = Sim(World(HOME + "/workshop.py")); sim.start()

        class Logged(make_handler(sim)):
            def log_message(self, fmt, *args):
                seen.append("%s %s" % (self.headers.get("Host"), fmt % args))

        httpd = ThreadingHTTPServer((host, port), Logged)
        threading.Thread(target=httpd.serve_forever, daemon=True).start()

    # a: python's own test server; b: garden open to this machine only; c: garden open to all
    subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--bind", "0.0.0.0"], cwd="/content",
                     stdout=open("/content/diag_a.log", "w"), stderr=subprocess.STDOUT)
    logging_garden("127.0.0.1", 8770, diag["B"])
    logging_garden("0.0.0.0", 8780, diag["C"])
    time.sleep(1)
    for name, p in (("A", 8000), ("B", 8770), ("C", 8780)):
        try:
            print(name, "answers inside colab:", urllib.request.urlopen("http://127.0.0.1:%d/" % p).status)
        except Exception as e:
            print(name, "does not answer inside colab:", e)
    print("browser:", output.eval_js("navigator.userAgent"))
    for name, p in (("A: test server", 8000), ("B: garden, this machine only", 8770), ("C: garden, open to all", 8780)):
        print("\n" + name, " own tab:", output.eval_js("google.colab.kernel.proxyPort(%d)" % p))
        output.serve_kernel_port_as_iframe(p, height=300)
else:
    print("A received:"); print(open("/content/diag_a.log").read() or "  nothing")
    for name in ("B", "C"):
        print(name, "received:"); print("\n".join(diag[name][-15:]) or "  nothing")